# USV Spectrograms — Plots

This notebook renders ultrasonic-vocalization (USV) spectrograms and the
embedding / summary figures derived from them, all driven from
`usv_playpen.visualizations.make_usv_spectrograms`. It runs **after** the
processing pipeline has produced, per session, a concatenated multi-channel
`*_int16.mmap*` audio file, a `*_usv_summary.csv` (per-USV
start / stop / duration / category / emitter), the 3D
translated / rotated / metric tracks, and — for the stitched
figures — the consolidated SAM2 + spectrogram HDF5 store.

The cells are organised as: *(1) imports*, *(2) parameters — every
tweakable knob*, *(3) derived setup*, then *(4) per-figure compute + plot*.
Each plotting cell is independent — you can re-run any single one without
re-running the rest, provided the **Imports**, **Parameters**, and
**Setup (derived)** cells have run.

The figures exposed here are:

- `USVSpectrogramPlotter` — single-channel / all-channel / stitched
  spectrogram modes for one session.
- `plot_usv_property_histograms` — pooled per-USV property histograms
  across many sessions.
- `plot_session_type_usv_counts` — mean USVs per session compared across
  the three session types (male-female / female-female / lone-male).
- `plot_session_usv_timeline` — every USV in one session drawn as a
  colored interval on a horizontal strip.

In [ ]:
### Imports

from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt

from usv_playpen.os_utils import configure_path
from usv_playpen.visualizations.plot_style import apply_plot_style
from usv_playpen.visualizations.auxiliary_plot_functions import create_colormap
from usv_playpen.visualizations.make_usv_spectrograms import (
    USVSpectrogramPlotter,
    plot_session_type_usv_counts,
    plot_session_usv_timeline,
    plot_usv_property_histograms,
)

apply_plot_style()

## Parameters

Every knob a user might tweak for a run lives in the single cell below —
data sources, the shared noise filter, output toggles, and all per-figure
styling / sampling options. Network-store paths are wrapped in
`os_utils.configure_path` so a path written with `/mnt/falkner/...` works
on macOS (auto-mapped to `/Volumes/falkner/...`) and vice-versa on Linux.
Edit here; nothing downstream redefines these.

In [ ]:
### Parameters — every user-tweakable knob lives here

# Shared noise filtering (applied to every per-USV summary CSV): rows whose
# `noise_col_id` value is in `noise_categories` are dropped before plotting.
noise_col_id = 'vae_supercategory'
noise_categories = (0,)

# 1. USV spectrogram plotter (single / all / stitched modes)
# A session directory containing a `*_int16.mmap*` audio file (and, for the
# stitched mode, a `*_usv_summary.csv` 1:1 with the consolidated h5 entries).
# All other knobs for this figure live in the `make_usv_spectrograms` block
# of visualizations_settings.json; override them on `vis_settings` in Setup.
spectrogram_session_root = configure_path('/mnt/falkner/Bartul/Data/20230124_192908')
# 'female' / 'male' -> use the per-sex colormap built in Setup; None -> fall
# back to the matplotlib name in
# vis_settings['make_usv_spectrograms']['spectrogram_cmap'].
spectrogram_cmap_choice = 'female'

# 2. Pooled USV property histograms
histograms_sessions_txt_path = configure_path('/mnt/falkner/Bartul/modeling/input_files/behavioral_courtship_intact_partners_sessions_list.txt')
histograms_output_path = None
histograms_fig_format = 'svg'

# 3. Mean USVs per session by session type
male_female_txt_path = configure_path('/mnt/falkner/Bartul/modeling/input_files/behavioral_courtship_intact_partners_sessions_list.txt')
female_female_txt_path = configure_path('/mnt/falkner/Bartul/modeling/input_files/behavioral_female_female_sessions_list.txt')
lone_male_txt_path = configure_path('/mnt/falkner/Bartul/modeling/input_files/ephys_lone_male_sessions_list.txt')
session_counts_output_path = configure_path('/Users/bmimica/Downloads/session_type_usv_counts')
session_counts_fig_format = 'svg'

# 4. Per-session USV timeline
timeline_session_root = configure_path('/mnt/falkner/Bartul/Data/20230119_172410')
timeline_window = (457, 462)              # (start_s, end_s); None shows the whole session
timeline_output_path = configure_path('/Users/bmimica/Downloads/session_usv_timeline_20230119_172410_457s-462s')
timeline_fig_format = 'svg'

## Setup (derived)

Objects derived from the parameters and from `visualizations_settings.json`
(the loaded settings dict, the per-sex sequential colormaps, and the
resolved spectrogram colormap override). You should not need to edit this
cell.

In [ ]:
### Setup (derived from the parameters above + visualizations_settings.json — do not edit)

with open(Path.cwd().parent / "_parameter_settings" / "visualizations_settings.json", 'r') as vis_settings_file:
    vis_settings = json.load(vis_settings_file)

male_color = vis_settings["male_colors"][0]
female_color = vis_settings["female_colors"][0]

# One sequential base-color -> white colormap per sex (identical settings, so
# build in a loop).
sex_cmaps = {}
for cmap_name, base_hex in (("female_cm", female_color), ("male_cm", male_color)):
    sex_cmaps[cmap_name] = create_colormap(input_parameter_dict={
        "cm_length": 255,
        "cm_name": cmap_name,
        "cm_type": "sequential",
        "cm_start": (int(base_hex[1:3], 16), int(base_hex[3:5], 16), int(base_hex[5:7], 16)),
        "cm_end": (255, 255, 255),
        "equalize_luminance": True,
        "match_luminance_by": "max",
        "change_saturation": 0.5,
        "cm_opacity": 1,
    })
female_cmap = sex_cmaps["female_cm"]
male_cmap = sex_cmaps["male_cm"]

# Resolve the spectrogram cmap override from the string choice in Parameters.
spectrogram_cmap_override = {"female": female_cmap, "male": male_cmap, None: None}[spectrogram_cmap_choice]

## 1. USV spectrogram plotter

`USVSpectrogramPlotter` renders USV spectrograms in three modes (set via
`vis_settings['make_usv_spectrograms']['mode']`):

- `"single"` — one channel's spectrogram, optionally with the raw waveform
  of that channel stacked above. dB amplitude scale.
- `"all"` — every channel's spectrogram in a single vertically stacked
  figure (with optional raw-waveform row per channel). dB amplitude scale.
- `"stitched"` — a session-timeline spectrogram built by placing the
  pre-computed `[0, 1]`-normalized per-USV spectrograms from the
  consolidated HDF5 store at their on-session start times. Gaps between
  USVs are exactly zero. Renders in **linear normalized amplitude** with a
  fixed `[0, 1]` colorbar (`cbar_limits` is ignored for this mode);
  requires `consolidated_spectrograms_h5` to point at the store, whose freq
  axis is fixed (~30–120 kHz linear, 128 bins).

The mode, channel, window (`time_window`, seconds; an `end` of `0` means
"end of recording"), `freq_limits` (kHz), `nfft`, colorbar and save knobs
all live in the `make_usv_spectrograms` block of
`visualizations_settings.json` (loaded into `vis_settings` in Setup —
override entries there before running). Pass `cmap_override` to use one of
the per-sex colormaps instead of the settings cmap string (resolved from
`spectrogram_cmap_choice` in Setup). If `save_fig` is `True`, the figure is
written under `save_dir` (defaults to `<session_root>/audio` when empty).

In [ ]:
### Plot USV spectrogram

plotter = USVSpectrogramPlotter(
    root_directory=spectrogram_session_root,
    visualizations_parameter_dict=vis_settings,
    message_output=print,
    cmap_override=spectrogram_cmap_override,
)

fig_spectrogram = plotter.make_usv_spectrograms()
plt.show()

## 2. Pooled USV property histograms

`plot_usv_property_histograms` is a module-level helper (not a method on
`USVSpectrogramPlotter`) that summarizes per-USV properties across many
sessions in a single figure. The session list, output path and noise
filter are set in the **Parameters** cell.

Five histograms are drawn, in panel order: `duration` (ms),
`mean_amplitude` (a.u.), `mean_freq_hz` (kHz), `freq_bandwidth_hz` (kHz),
`spectral_entropy` (a.u.). Each panel uses 36 linearly-spaced bins over the
FeatureZoo theoretical range, with `histtype='stepfilled'`, fill `#808080`,
edge `#000000`, top / right spines hidden. The title reports the number of
sessions successfully loaded and the total number of pooled vocalizations.

In [ ]:
### Plot pooled USV property histograms

fig_histograms = plot_usv_property_histograms(
    sessions_txt_path=histograms_sessions_txt_path,
    output_path=histograms_output_path,
    fig_format=histograms_fig_format,
    noise_col_id=noise_col_id,
    noise_categories=noise_categories,
)
plt.show()

## 3. Mean USVs per session by session type

`plot_session_type_usv_counts` takes **three** session-list txt files — one
each for male-female, female-female and lone-male sessions (set in
**Parameters**) — and renders a horizontal bar chart comparing the mean
number of non-noise USVs per session, with SEM error bars
(`std(counts, ddof=1) / sqrt(n_sessions)`) on each bar.

Bar colors: lone male → `#9AC0CD` (project male color), female-female →
`#FF6347` (project female color), male-female → `#DFB8B3` (a pastel
midpoint mixed 35% toward white). For each session list the function
discovers `*_usv_summary.csv`, drops noise rows, and counts the rest;
sessions whose CSV is missing or unreadable (SMB timeouts etc.) are logged
and excluded from that group's mean.

In [ ]:
### Plot mean USVs per session by session type

fig_session_counts = plot_session_type_usv_counts(
    male_female_txt_path=male_female_txt_path,
    female_female_txt_path=female_female_txt_path,
    lone_male_txt_path=lone_male_txt_path,
    output_path=session_counts_output_path,
    fig_format=session_counts_fig_format,
    noise_col_id=noise_col_id,
    noise_categories=noise_categories,
)
plt.show()

## 4. Per-session USV timeline

`plot_session_usv_timeline` renders every non-noise USV in one session as a
colored rectangle spanning its `[start, stop]` interval on a single
horizontal strip. The session, optional `timeline_window` (a
`(start_s, end_s)` clip; `None` shows the whole session), output path and
noise filter are set in **Parameters**.

The session's male / female track ids are read from
`<session>/video/*_points3d_translated_rotated_metric.h5`
(`track_names[0]` = male, `track_names[1]` = female — same convention as
`usv_summary_statistics.extract_session_metadata`). Each USV's CSV
`emitter` field is matched against these ids: male id → `#9AC0CD`, female id
→ `#FF6347`, anything else → `#C0C0C0` (unassigned). The title reports the
total non-noise count.

In [ ]:
### Plot per-session USV timeline

fig_timeline = plot_session_usv_timeline(
    session_root=timeline_session_root,
    time_window=timeline_window,
    output_path=timeline_output_path,
    fig_format=timeline_fig_format,
    noise_col_id=noise_col_id,
    noise_categories=noise_categories,
)
plt.show()